<a href="https://colab.research.google.com/github/odalishdz071/detective-de-senales/blob/fase-1-datos/detective-fase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1

In [ ]:
# 1.1 Configuración del entorno
# Librerías de cómputo numérico
import numpy as np

# Librerías de procesamiento de señales
from scipy import signal

# Librerías de visualización
import matplotlib.pyplot as plt

# Librerías de machine learning (división de datos)
from sklearn.model_selection import train_test_split

# Para guardar/cargar el dataset
import pickle

# Semilla para reproducibilidad
np.random.seed(42)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [ ]:
# 1.2 Definir los parámetros de la señal
# 1. Frecuencia de muestreo (fs)
fs = 10000 # Hz (10 kHz)

# 2. Frecuencia portadora (fc)
fc = 2000 # Hz (2 kHz)

# 3. Tasa de símbolos (Rs)
Rs = 100 # Baudios (símbolos por segundo)

# Cálculos derivados
Tb = 1 / Rs # Tiempo de duración de un símbolo (0.01 s)
n_symbols = 20 # Cantidad de símbolos que tendrá cada señal
n_samples = int(n_symbols * Tb * fs) # Total de muestras por señal (2000 muestras)

print(f"- Parámetros Configurados -")
print(f"Frecuencia de muestreo (fs): {fs} Hz")
print(f"Frecuencia portadora (fc): {fc} Hz")
print(f"Tasa de símbolos (Rs): {Rs} Baudios")
print()
print(f"Duración de cada símbolo (Tb): {Tb} segundos")
print(f"Muestras por señal: {n_samples}")

# Verificación rápida de Nyquist
if fs >= 2 * fc:
    print("Cumple con el Teorema de Nyquist")
else:
    print("Alerta: Posible Aliasing")

- Parámetros Configurados -
Frecuencia de muestreo (fs): 10000 Hz
Frecuencia portadora (fc): 2000 Hz
Tasa de símbolos (Rs): 100 Baudios

Duración de cada símbolo (Tb): 0.01 segundos
Muestras por señal: 2000
Cumple con el Teorema de Nyquist


In [ ]:
# 1.3 Generadores de señales
def generar_bits(n_bits, bits_por_simbolo=1): # Genera bits aleatorios (0, 1)
    return np.random.randint(0, 2, n_bits * bits_por_simbolo)

def pulso_nrz(bits, fs, Rs, n_samples): # Convierte bits en forma de onda NRZ
    muestras_por_simbolo = int(fs / Rs)
    señal = np.repeat(bits, muestras_por_simbolo)
    return señal[:n_samples]

# ASK: bit 1 (amplitud A), bit 0 (amplitud 0)
def modulacion_ask(bits, fs, fc, Rs, n_samples, A=1.0):
    t = np.arange(n_samples) / fs
    portadora = np.cos(2 * np.pi * fc * t)
    envolvente = pulso_nrz(bits, fs, Rs, n_samples)
    return A * envolvente * portadora

# FSK: bit 1 (fc + delta_f), bit 0 (fc - delta_f)
def modulacion_fsk(bits, fs, fc, Rs, n_samples, delta_f=500):
    t = np.arange(n_samples) / fs
    muestras_por_simbolo = int(fs / Rs)
    frecuencias = np.where(bits == 1, fc + delta_f, fc - delta_f)
    freq_inst = np.repeat(frecuencias, muestras_por_simbolo)[:n_samples]
    fase = 2 * np.pi * np.cumsum(freq_inst) / fs
    return np.cos(fase)

# BPSK: bit 1 (fase 0), bit 0 (fase pi = 180 grados)
def modulacion_psk(bits, fs, fc, Rs, n_samples):
    t = np.arange(n_samples) / fs
    fases = np.where(bits == 1, 0, np.pi)
    fase_inst = np.repeat(fases, int(fs / Rs))[:n_samples]
    return np.cos(2 * np.pi * fc * t + fase_inst)

# 16-QAM: 4 bits por símbolo (Combinación de Amplitud y Fase)
def modulacion_qam16(bits, fs, fc, Rs, n_samples):
    muestras_por_simbolo = int(fs / Rs)
    n_simbolos = n_samples // muestras_por_simbolo
    # Tomamos los bits y los agrupamos de 4 en 4
    bits = bits[:n_simbolos * 4].reshape(-1, 4)

    # Mapeo de 4 bits a niveles de I y Q (2 bits para I, 2 bits para Q)
    I = 2 * bits[:, 0] + bits[:, 1] - 1.5   # Niveles: -1.5, -0.5, 0.5, 1.5
    Q = 2 * bits[:, 2] + bits[:, 3] - 1.5

    # Normalizar para que la potencia promedio sea 1
    I = I / np.sqrt(5)
    Q = Q / np.sqrt(5)

    # Repetir cada símbolo para que dure el tiempo de símbolo
    I_rep = np.repeat(I, muestras_por_simbolo)[:n_samples]
    Q_rep = np.repeat(Q, muestras_por_simbolo)[:n_samples]
    t = np.arange(n_samples) / fs
    # Ecuación de QAM: I*cos - Q*sin
    return I_rep * np.cos(2 * np.pi * fc * t) - Q_rep * np.sin(2 * np.pi * fc * t)

print("Funciones de modulación creadas y listas para usarse.")

Funciones de modulación creadas y listas para usarse.


In [ ]:
# 1.4 Generar el conjunto de datos
import time # Para medir cuánto tarda
N_POR_CLASE = 1000 # 1000 muestras por cada modulación
# Diccionario para asignar una etiqueta a cada modulación
modulaciones = {
    'ASK': 0, 'FSK': 1, 'PSK': 2, 'QAM16': 3
}
X = [] # Aquí guardaremos las formas de onda (las señales)
y = [] # Aquí guardaremos las etiquetas (0, 1, 2 o 3)
inicio = time.time()
print("Generando datos...")
for nombre, etiqueta in modulaciones.items():
    print(f" -> Generando 1000 señales de {nombre}...")
    for _ in range(N_POR_CLASE):
        if nombre == 'QAM16':
            # QAM usa 4 bits por símbolo
            bits = generar_bits(n_symbols, bits_por_simbolo=4)
            señal = modulacion_qam16(bits, fs, fc, Rs, n_samples)
        else:
            # ASK, FSK y PSK usan 1 bit por símbolo
            bits = generar_bits(n_symbols, bits_por_simbolo=1)
            if nombre == 'ASK':
                señal = modulacion_ask(bits, fs, fc, Rs, n_samples)
            elif nombre == 'FSK':
                señal = modulacion_fsk(bits, fs, fc, Rs, n_samples)
            elif nombre == 'PSK':
                señal = modulacion_psk(bits, fs, fc, Rs, n_samples)
        # Guardamos la señal y su etiqueta en las listas
        X.append(señal)
        y.append(etiqueta)
# Convertimos las listas a arreglos de NumPy (Matrices)
X = np.array(X)
y = np.array(y)
fin = time.time()
print(f"\n Dataset generado en {fin - inicio:.2f} segundos")
print("Forma de X (señales):", X.shape)
print("Forma de y (etiquetas):", y.shape)
print("Distribución de clases:", np.bincount(y))

Generando datos...
 -> Generando 1000 señales de ASK...
 -> Generando 1000 señales de FSK...
 -> Generando 1000 señales de PSK...
 -> Generando 1000 señales de QAM16...

 Dataset generado en 0.40 segundos
Forma de X (señales): (4000, 2000)
Forma de y (etiquetas): (4000,)
Distribución de clases: [1000 1000 1000 1000]


In [ ]:
# 1.5 Añadir Ruido (AWGN)
def agregar_awgn(señal, snr_db): # Añade Ruido Gaussiano a una señal
    # 1. Calculamos la potencia de la señal original
    potencia_señal = np.mean(señal ** 2)

    # 2. Calculamos la potencia del ruido según el SNR deseado
    potencia_ruido = potencia_señal / (10 ** (snr_db / 10))

    # 3. Generamos el ruido con una distribución normal (gaussiana)
    ruido = np.random.normal(0, np.sqrt(potencia_ruido), señal.shape)

    # 4. Sumamos el ruido a la señal original
    return señal + ruido

print("Generando dataset con ruido aleatorio para entrenamiento...")

# Usamos un SNR aleatorio entre -5 dB y 20 dB para entrenar a la IA
X_ruidoso = np.array([agregar_awgn(s, np.random.uniform(-5, 20)) for s in X])

# Generamos versiones de prueba a niveles de SNR fijos que nos servirán para la Fase 3
snr_niveles = [-10, -5, 0, 5, 10, 15, 20]
X_snr = {}
for snr in snr_niveles:
    X_snr[snr] = np.array([agregar_awgn(s, snr) for s in X])
print("Ruido AWGN aplicado correctamente.")
print("Forma del dataset ruidoso (entrenamiento):", X_ruidoso.shape)
print("Niveles de SNR preparados para pruebas:", snr_niveles)

Generando dataset con ruido aleatorio para entrenamiento...
Ruido AWGN aplicado correctamente.
Forma del dataset ruidoso (entrenamiento): (4000, 2000)
Niveles de SNR preparados para pruebas: [-10, -5, 0, 5, 10, 15, 20]


In [ ]:
# 1.6 Separación de datos (Train/Test Split)
# Dividimos el dataset ruidoso (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_ruidoso, y, test_size=0.2, random_state=42, stratify=y
)

# También guardamos las versiones limpias de prueba
_, X_test_clean, _, y_test_clean = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("- Resumen de la División de Datos -")
print("Datos de Entrenamiento (X_train):", X_train.shape)
print("Etiquetas de Entrenamiento (y_train):", y_train.shape)
print("Datos de Prueba (X_test):", X_test.shape)
print("Etiquetas de Prueba (y_test):", y_test.shape)

# Guardar el dataset en un archivo para no tener que generarlo de nuevo
np.savez_compressed('dataset_detective.npz',
    X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test,
    X_test_clean=X_test_clean, y_test_clean=y_test_clean
)
print("Dataset guardado exitosamente en 'dataset_detective.npz'")

- Resumen de la División de Datos -
Datos de Entrenamiento (X_train): (3200, 2000)
Etiquetas de Entrenamiento (y_train): (3200,)
Datos de Prueba (X_test): (800, 2000)
Etiquetas de Prueba (y_test): (800,)
Dataset guardado exitosamente en 'dataset_detective.npz'
